# Рендер Blender в Google Colab

Этот notebook пошагово рендерит уже подготовленный проект и не сохраняет изменения в исходный `.blend`. Выполняйте главы сверху вниз; после обрыва runtime повторите загрузку, preflight и главу render с тем же `JOB_ID`.

## Перед стартом

1. Для Cycles включите `Runtime → Change runtime type → GPU`.
2. Упакуйте внешние ресурсы через `File → External Data → Pack Resources` **или** подготовьте ZIP с исходной структурой папок.
3. Загрузите ровно один `.blend` либо один ZIP-проект.

<details><summary>Что notebook не делает</summary>

Он не меняет сохранённые scene settings, не сохраняет исходный `.blend`, не запускает Colab автоматически и не хранит проект, результаты или credentials в Blender cache.

</details>

## Настройки

Основные параметры выше подходят большинству запусков. Сохранённые в проекте настройки сцены остаются источником истины.

In [ ]:
#@title Настройки — основные
# Выберите LTS-версию. `custom` разрешает только явную версию X.Y.Z.
BLENDER_VERSION_PRESET = '5.2.1' #@param ['5.2.1', '4.5.13', 'custom']
# True: выбрать GPU средствами Blender/Cycles.
ENABLE_CYCLES_GPU = True #@param {type: 'boolean'}
RENDER_MODE = 'ANIMATION' #@param ['ANIMATION', 'STILL']
STILL_FRAME = 1 #@param {type: 'integer'}
from dataclasses import dataclass
from enum import Enum
import re

LTS_VERSION_PRESETS = ('5.2.1', '4.5.13')
CUSTOM_VERSION_PRESET = 'custom'
VERSION_RE = re.compile(r'^(0|[1-9]\d*)\.(0|[1-9]\d*)\.(0|[1-9]\d*)$')

class ConfigurationError(ValueError):
    pass

class RenderMode(str, Enum):
    STILL = 'STILL'
    ANIMATION = 'ANIMATION'

def validate_blender_version(version):
    if not isinstance(version, str) or not VERSION_RE.fullmatch(version.strip()):
        raise ConfigurationError('Версия Blender должна быть точной строкой X.Y.Z, например 5.2.1.')
    return version.strip()

def resolve_blender_version(preset, custom_version):
    if not isinstance(preset, str) or not isinstance(custom_version, str):
        raise ConfigurationError('Значения версии должны быть строками.')
    preset = preset.strip()
    if preset == CUSTOM_VERSION_PRESET:
        if not custom_version.strip():
            raise ConfigurationError('Для preset custom заполните CUSTOM_BLENDER_VERSION.')
        return validate_blender_version(custom_version)
    if preset not in LTS_VERSION_PRESETS:
        raise ConfigurationError('Выберите LTS preset или custom.')
    if custom_version.strip():
        raise ConfigurationError('CUSTOM_BLENDER_VERSION разрешён только для preset custom.')
    return preset

@dataclass(frozen=True)
class RenderConfig:
    blender_version: str
    enable_cycles_gpu: bool
    allow_cpu_fallback: bool
    include_cpu_with_gpu: bool
    render_mode: RenderMode
    still_frame: int
    download_result: bool
    run_cycles_smoke_test: bool
    run_preflight_test_frame: bool
    enable_drive_blender_cache: bool

def validate_config():
    values = {
        'ENABLE_CYCLES_GPU': ENABLE_CYCLES_GPU,
        'ALLOW_CPU_FALLBACK': ALLOW_CPU_FALLBACK,
        'INCLUDE_CPU_WITH_GPU': INCLUDE_CPU_WITH_GPU,
        'DOWNLOAD_RESULT': DOWNLOAD_RESULT,
        'RUN_CYCLES_SMOKE_TEST': RUN_CYCLES_SMOKE_TEST,
        'RUN_PREFLIGHT_TEST_FRAME': RUN_PREFLIGHT_TEST_FRAME,
        'ENABLE_DRIVE_BLENDER_CACHE': ENABLE_DRIVE_BLENDER_CACHE,
    }
    for name, value in values.items():
        if not isinstance(value, bool):
            raise ConfigurationError(f'{name} должен быть True или False.')
    if isinstance(STILL_FRAME, bool) or not isinstance(STILL_FRAME, int) or STILL_FRAME < 0:
        raise ConfigurationError('STILL_FRAME должен быть неотрицательным целым числом.')
    try:
        render_mode = RenderMode(RENDER_MODE.strip().upper())
    except (AttributeError, ValueError) as error:
        raise ConfigurationError('RENDER_MODE должен быть STILL или ANIMATION.') from error
    if INCLUDE_CPU_WITH_GPU and not ENABLE_CYCLES_GPU:
        raise ConfigurationError('INCLUDE_CPU_WITH_GPU требует ENABLE_CYCLES_GPU=True.')
    return RenderConfig(
        blender_version=resolve_blender_version(BLENDER_VERSION_PRESET, CUSTOM_BLENDER_VERSION),
        enable_cycles_gpu=ENABLE_CYCLES_GPU,
        allow_cpu_fallback=ALLOW_CPU_FALLBACK,
        include_cpu_with_gpu=INCLUDE_CPU_WITH_GPU,
        render_mode=render_mode,
        still_frame=STILL_FRAME,
        download_result=DOWNLOAD_RESULT,
        run_cycles_smoke_test=RUN_CYCLES_SMOKE_TEST,
        run_preflight_test_frame=RUN_PREFLIGHT_TEST_FRAME,
        enable_drive_blender_cache=ENABLE_DRIVE_BLENDER_CACHE,
    )

# Выполните следующую сворачиваемую Advanced-главу: она задаёт безопасные значения
# по умолчанию и затем завершает проверку CONFIG.

<details><summary>Advanced — меняйте только при осознанной необходимости</summary>

Здесь находятся custom version, CPU fallback, diagnostic smoke test, test frame, скачивание результата и opt-in Drive cache. Все значения по умолчанию безопасны: CPU fallback выключен, cache выключен.

</details>

In [ ]:
#@title Настройки — Advanced
CUSTOM_BLENDER_VERSION = '' #@param {type: 'string'}
ALLOW_CPU_FALLBACK = False #@param {type: 'boolean'}
INCLUDE_CPU_WITH_GPU = False #@param {type: 'boolean'}
DOWNLOAD_RESULT = False #@param {type: 'boolean'}
RUN_CYCLES_SMOKE_TEST = False #@param {type: 'boolean'}
RUN_PREFLIGHT_TEST_FRAME = False #@param {type: 'boolean'}

# Cache хранит только проверенный архив Blender в отдельной папке Drive.
ENABLE_DRIVE_BLENDER_CACHE = False #@param {type: 'boolean'}

CONFIG = validate_config()
BLENDER_VERSION = CONFIG.blender_version
print(f'Blender: {BLENDER_VERSION}; Cycles GPU: {CONFIG.enable_cycles_gpu}; CPU fallback: {CONFIG.allow_cpu_fallback}; Drive cache: {CONFIG.enable_drive_blender_cache}')

## Blender и GPU

Архив Blender всегда сверяется с SHA-256 из `download.blender.org`. При включённом cache Drive авторизуется только для отдельной папки с этим архивом; Blender распаковывается и запускается только из `/content`.

In [ ]:
#@title Установка Blender с официальной SHA-256 проверкой
import hashlib
from pathlib import Path
import shutil
import subprocess
import tempfile
from urllib.request import Request, urlopen

blender_series = '.'.join(BLENDER_VERSION.split('.')[:2])
archive_name = f'blender-{BLENDER_VERSION}-linux-x64.tar.xz'
release_url = f'https://download.blender.org/release/Blender{blender_series}'
archive_url = f'{release_url}/{archive_name}'
checksum_url = f'{release_url}/blender-{BLENDER_VERSION}.sha256'
install_root = Path('/content/blender')
blender_dir = install_root / f'blender-{BLENDER_VERSION}-linux-x64'
BLENDER = blender_dir / 'blender'
archive_path = install_root / archive_name
DRIVE_BLENDER_CACHE_ROOT = Path('/content/drive/MyDrive/Blend-to-Colab/blender-cache')
cache_path = DRIVE_BLENDER_CACHE_ROOT / archive_name if CONFIG.enable_drive_blender_cache else None

def checksum_for_archive(manifest, filename):
    for line in manifest.splitlines():
        fields = line.strip().split(maxsplit=1)
        if len(fields) == 2 and fields[1].lstrip('*') == filename:
            if re.fullmatch(r'[0-9a-fA-F]{64}', fields[0]):
                return fields[0].lower()
    raise RuntimeError(f'Официальный checksum manifest не содержит {filename}.')

def sha256_file(path):
    digest = hashlib.sha256()
    with path.open('rb') as source:
        for chunk in iter(lambda: source.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

OFFICIAL_DOWNLOAD_USER_AGENT = 'Blender-to-GoogleColab/phase-1'

def open_official_url(url):
    return urlopen(Request(url, headers={'User-Agent': OFFICIAL_DOWNLOAD_USER_AGENT}))

def atomic_copy_verified_archive(source, destination, expected_checksum):
    if not source.is_file() or sha256_file(source) != expected_checksum:
        raise RuntimeError('Архив Blender не совпадает с официальным SHA-256.')
    destination.parent.mkdir(parents=True, exist_ok=True)
    with tempfile.NamedTemporaryFile(dir=destination.parent, prefix=f'.{destination.name}.', suffix='.tmp', delete=False) as temporary:
        temporary_path = Path(temporary.name)
    try:
        shutil.copyfile(source, temporary_path)
        if sha256_file(temporary_path) != expected_checksum:
            raise RuntimeError('Архив Blender изменился во время копирования.')
        temporary_path.replace(destination)
    finally:
        temporary_path.unlink(missing_ok=True)

def download_verified_archive(destination, expected_checksum):
    with tempfile.NamedTemporaryFile(dir=destination.parent, prefix=f'.{archive_name}.', delete=False) as temporary:
        temporary_path = Path(temporary.name)
        try:
            with open_official_url(archive_url) as response:
                shutil.copyfileobj(response, temporary)
            actual_checksum = sha256_file(temporary_path)
            if actual_checksum != expected_checksum:
                raise RuntimeError('SHA-256 Blender archive не совпал; частичная загрузка удалена.')
            temporary_path.replace(destination)
        except Exception:
            temporary_path.unlink(missing_ok=True)
            raise

# Проверяем официальный manifest до каждого reuse cache entry.
with open_official_url(checksum_url) as response:
    expected_checksum = checksum_for_archive(response.read().decode('utf-8'), archive_name)

if cache_path is not None:
    from google.colab import drive
    drive.mount('/content/drive')
    print(f'Opt-in Drive cache: {cache_path}')

if not BLENDER.exists():
    subprocess.run(['apt-get', 'update', '-qq'], check=True)
    subprocess.run(['apt-get', 'install', '-y', '-qq', 'libgl1', 'libglib2.0-0', 'libsm6'], check=True)
    install_root.mkdir(parents=True, exist_ok=True)
    cache_status = 'miss' if cache_path is not None else 'disabled'
    if cache_path is not None and cache_path.exists():
        try:
            atomic_copy_verified_archive(cache_path, archive_path, expected_checksum)
            cache_status = 'hit'
        except (RuntimeError, OSError):
            cache_status = 'corrupt; downloading a fresh official archive'
    if cache_status != 'hit':
        download_verified_archive(archive_path, expected_checksum)
        if cache_path is not None:
            try:
                atomic_copy_verified_archive(archive_path, cache_path, expected_checksum)
            except (RuntimeError, OSError):
                cache_status = 'write_failed; verified local archive will still be used'
    print(f'Blender archive cache: {cache_status}')
    subprocess.run(['tar', '-xf', str(archive_path), '-C', str(install_root)], check=True)
else:
    print('Blender уже распакован в локальном /content; Drive cache не используется для запуска.')

if not BLENDER.exists():
    raise RuntimeError(f'Blender не найден после распаковки: {BLENDER}')
subprocess.run([str(BLENDER), '--version'], check=True)

In [ ]:
#@title 2.1. Реальный Cycles GPU smoke test (опционально)
# При RUN_CYCLES_SMOKE_TEST=True создаётся новая временная сцена. Загруженный проект не открывается и не изменяется.
if CONFIG.run_cycles_smoke_test:
    smoke_script = Path('/content/cycles_gpu_smoke_test.py')
    smoke_log = Path('/content/cycles_gpu_smoke_test.log')
    smoke_script.write_text('''import bpy
import math

def refresh_devices(preferences):
    refresh = getattr(preferences, 'refresh_devices', None)
    if refresh is not None:
        refresh()
    else:
        preferences.get_devices()

addon = bpy.context.preferences.addons.get('cycles')
if addon is None:
    raise RuntimeError('Cycles add-on/preferences are unavailable.')
preferences = addon.preferences
selected_backend = None
selected_devices = []
errors = []
for backend in ('OPTIX', 'CUDA'):
    try:
        preferences.compute_device_type = backend
        refresh_devices(preferences)
        devices = tuple(preferences.devices)
        selected_devices = [device for device in devices if str(device.type).upper() == backend]
        if not selected_devices:
            errors.append(f'{backend}: no matching Cycles devices')
            continue
        for device in devices:
            device.use = str(device.type).upper() == backend
        selected_backend = backend
        break
    except Exception as error:
        errors.append(f'{backend}: {error}')
if selected_backend is None:
    raise RuntimeError('Cycles GPU smoke test requires GPU; ' + '; '.join(errors))

scene = bpy.context.scene
scene.render.engine = 'CYCLES'
scene.cycles.device = 'GPU'
scene.cycles.samples = 1
scene.render.resolution_x = 64
scene.render.resolution_y = 64
scene.render.resolution_percentage = 100
bpy.ops.mesh.primitive_cube_add()
bpy.ops.object.camera_add(location=(0, -3, 0))
camera = bpy.context.object
camera.rotation_euler = (math.radians(90), 0, 0)
scene.camera = camera
bpy.ops.object.light_add(type='POINT', location=(2, -2, 3))
bpy.context.object.data.energy = 1000
bpy.ops.render.render(write_still=False)
names = [f'{device.name} ({device.type})' for device in selected_devices]
print(f'CYCLES_SMOKE_TEST_PASS backend={selected_backend}; devices={names}')
''', encoding='utf-8')
    result = subprocess.run(
        [str(BLENDER), '--background', '--factory-startup', '--python', str(smoke_script)],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    )
    smoke_log.write_text(result.stdout, encoding='utf-8')
    print(result.stdout)
    if result.returncode != 0:
        raise RuntimeError(f'Cycles GPU smoke test failed; log: {smoke_log}')
    if 'CYCLES_SMOKE_TEST_PASS backend=' not in result.stdout:
        raise RuntimeError(f'Cycles smoke test produced no GPU confirmation; log: {smoke_log}')
    print(f'Cycles GPU smoke test confirmed by Blender; log: {smoke_log}')
else:
    print('Cycles GPU smoke test is prepared. Set RUN_CYCLES_SMOKE_TEST=True and rerun this cell in a GPU Colab runtime.')

## Загрузка проекта

Загрузите один `.blend` или один ZIP. ZIP проверяется до распаковки: traversal, symbolic links, archive bombs и нехватка места блокируются. Исходный `.blend` остаётся только локальным staging input.

In [ ]:
#@title Безопасная загрузка `.blend` или ZIP-проекта
from google.colab import files
from pathlib import Path, PurePosixPath, PureWindowsPath
import shutil
import stat
import zipfile
from urllib.parse import unquote, urlsplit
from urllib.request import Request, urlopen

MAX_ARCHIVE_ENTRIES = 10_000
MAX_ARCHIVE_UNCOMPRESSED_BYTES = 25 * 1024 ** 3
MAX_ARCHIVE_COMPRESSION_RATIO = 100

class ArchiveSafetyError(ValueError):
    pass

def validate_zip_member(info):
    name = info.filename
    posix_path = PurePosixPath(name)
    windows_path = PureWindowsPath(name)
    if not name or '\x00' in name:
        raise ArchiveSafetyError('ZIP содержит пустое или NUL-имя entry.')
    if posix_path.is_absolute() or windows_path.is_absolute() or windows_path.drive:
        raise ArchiveSafetyError(f'ZIP содержит абсолютный путь: {name!r}')
    if any(part == '..' for part in posix_path.parts):
        raise ArchiveSafetyError(f'ZIP содержит path traversal: {name!r}')
    mode = info.external_attr >> 16
    file_type = stat.S_IFMT(mode)
    if file_type and not (stat.S_ISREG(mode) or stat.S_ISDIR(mode)):
        raise ArchiveSafetyError(f'ZIP содержит special file или symlink: {name!r}')

def safe_extract_zip(archive_path, destination):
    with zipfile.ZipFile(archive_path) as archive:
        infos = tuple(archive.infolist())
        if len(infos) > MAX_ARCHIVE_ENTRIES:
            raise ArchiveSafetyError(f'Слишком много entries: {len(infos)} > {MAX_ARCHIVE_ENTRIES}.')
        total_uncompressed = 0
        for info in infos:
            validate_zip_member(info)
            if info.is_dir():
                continue
            if info.file_size and info.compress_size == 0:
                raise ArchiveSafetyError(f'Некорректный compressed size: {info.filename!r}')
            if info.file_size / max(info.compress_size, 1) > MAX_ARCHIVE_COMPRESSION_RATIO:
                raise ArchiveSafetyError(f'Подозрительный compression ratio: {info.filename!r}')
            total_uncompressed += info.file_size
            if total_uncompressed > MAX_ARCHIVE_UNCOMPRESSED_BYTES:
                raise ArchiveSafetyError('Распакованный размер ZIP превышает лимит безопасности.')
        destination.mkdir(parents=True, exist_ok=True)
        if destination.is_symlink():
            raise ArchiveSafetyError('Каталог распаковки не может быть symbolic link.')
        if total_uncompressed > shutil.disk_usage(destination).free:
            raise ArchiveSafetyError('Недостаточно свободного места для безопасной распаковки.')
        root = destination.resolve()
        for info in infos:
            target = root.joinpath(*PurePosixPath(info.filename).parts)
            try:
                target.resolve().relative_to(root)
            except ValueError as error:
                raise ArchiveSafetyError(f'ZIP entry выходит за staging: {info.filename!r}') from error
            if info.is_dir():
                target.mkdir(parents=True, exist_ok=True)
                continue
            target.parent.mkdir(parents=True, exist_ok=True)
            with archive.open(info) as source, target.open('xb') as output:
                shutil.copyfileobj(source, output)

#@title Выбор источника проекта
PROJECT_SOURCE = 'upload' #@param ['upload', 'drive_file', 'drive_folder', 'url']
PROJECT_SOURCE_PATH = '' #@param {type: 'string'}
PROJECT_URL = '' #@param {type: 'string'}
# При нескольких .blend укажите путь относительно staged project, например scenes/main.blend.
SELECTED_BLEND_PATH = '' #@param {type: 'string'}
MAX_PROJECT_INPUT_BYTES = 25 * 1024 ** 3
MAX_PROJECT_FILES = 10_000
DRIVE_MYDRIVE_ROOT = Path('/content/drive/MyDrive')

def require_https_url(value):
    if not isinstance(value, str) or not value.strip():
        raise RuntimeError('PROJECT_URL должен быть непустым HTTPS URL.')
    parsed = urlsplit(value.strip())
    if parsed.scheme != 'https' or not parsed.hostname or parsed.username or parsed.password or any(character.isspace() or ord(character) < 32 for character in value):
        raise RuntimeError('PROJECT_URL должен быть HTTPS URL без credentials, пробелов и control characters.')
    return value.strip(), Path(unquote(parsed.path)).name

def drive_path(value):
    if not isinstance(value, str) or not value.strip():
        raise RuntimeError('Для Drive source заполните PROJECT_SOURCE_PATH.')
    root = DRIVE_MYDRIVE_ROOT.resolve()
    candidate = Path(value.strip())
    candidate = candidate if candidate.is_absolute() else root / candidate
    candidate = candidate.resolve()
    try:
        candidate.relative_to(root)
    except ValueError as error:
        raise RuntimeError('Drive source должен находиться внутри MyDrive.') from error
    return candidate

def copy_drive_folder(source, destination):
    files_seen, bytes_seen = 0, 0
    for entry in source.rglob('*'):
        if entry.is_symlink() or not (entry.is_dir() or entry.is_file()):
            raise RuntimeError(f'Drive folder содержит недопустимый entry: {entry.relative_to(source)}')
        if entry.is_file():
            files_seen += 1; bytes_seen += entry.stat().st_size
            if files_seen > MAX_PROJECT_FILES or bytes_seen > MAX_PROJECT_INPUT_BYTES:
                raise RuntimeError('Drive folder превышает безопасный лимит файлов или размера.')
    shutil.copytree(source, destination, copy_function=shutil.copy2)

def download_https_project(url, destination):
    total = 0
    with urlopen(Request(url, headers={'User-Agent': 'Blender-to-GoogleColab/project-input'}), timeout=60) as response:
        require_https_url(response.geturl())
        declared_size = response.headers.get('Content-Length')
        if declared_size and int(declared_size) > MAX_PROJECT_INPUT_BYTES:
            raise RuntimeError('Remote project превышает безопасный лимит размера.')
        with destination.open('xb') as output:
            for chunk in iter(lambda: response.read(1024 * 1024), b''):
                total += len(chunk)
                if total > MAX_PROJECT_INPUT_BYTES:
                    raise RuntimeError('Remote project превысил безопасный лимит размера.')
                output.write(chunk)

def select_blend(project_root):
    root = project_root.resolve()
    candidates = tuple(sorted(path for path in root.rglob('*.blend') if path.is_file() and not path.is_symlink()))
    if len(candidates) == 1 and not SELECTED_BLEND_PATH.strip():
        return candidates[0]
    if not SELECTED_BLEND_PATH.strip():
        options = ', '.join(path.relative_to(root).as_posix() for path in candidates) or 'none'
        raise RuntimeError(f'Найдено .blend-файлов: {len(candidates)}. Укажите SELECTED_BLEND_PATH: {options}')
    relative = PurePosixPath(SELECTED_BLEND_PATH.replace('\\', '/'))
    if relative.is_absolute() or '..' in relative.parts or PureWindowsPath(SELECTED_BLEND_PATH).drive:
        raise RuntimeError('SELECTED_BLEND_PATH должен быть относительным безопасным путём.')
    selected = root.joinpath(*relative.parts).resolve()
    try:
        selected.relative_to(root)
    except ValueError as error:
        raise RuntimeError('SELECTED_BLEND_PATH выходит за staging project.') from error
    if selected not in candidates:
        raise RuntimeError('SELECTED_BLEND_PATH не указывает на staged .blend.')
    return selected

BLEND_CONTAINER_MAGICS = (b'BLENDER', b'\x28\xb5\x2f\xfd', b'\x1f\x8b')

def validate_blend(path):
    if path.is_symlink() or not path.is_file() or not 12 <= path.stat().st_size <= MAX_PROJECT_INPUT_BYTES:
        raise RuntimeError('Выбранный .blend не является допустимым regular file.')
    with path.open('rb') as source:
        container_magic = source.read(7)
        if not any(container_magic.startswith(magic) for magic in BLEND_CONTAINER_MAGICS):
            raise RuntimeError('Выбранный .blend не содержит поддерживаемый Blender container header.')

if PROJECT_SOURCE not in {'upload', 'drive_file', 'drive_folder', 'url'}:
    raise RuntimeError('PROJECT_SOURCE должен быть upload, drive_file, drive_folder или url.')
workspace = Path('/content/blender_project')
shutil.rmtree(workspace, ignore_errors=True)
uploads_dir = workspace / 'uploads'
project_dir = workspace / 'project'
uploads_dir.mkdir(parents=True)

if PROJECT_SOURCE == 'upload':
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise RuntimeError('Загрузите ровно один .blend или ZIP-архив проекта.')
    name, data = next(iter(uploaded.items()))
    uploaded_name = Path(name).name
    uploaded_path = uploads_dir / uploaded_name
    uploaded_path.write_bytes(data)
elif PROJECT_SOURCE == 'url':
    url, uploaded_name = require_https_url(PROJECT_URL)
    uploaded_path = uploads_dir / uploaded_name
    download_https_project(url, uploaded_path)
elif PROJECT_SOURCE == 'drive_file':
    from google.colab import drive
    drive.mount('/content/drive')
    source = drive_path(PROJECT_SOURCE_PATH)
    uploaded_name = source.name
    uploaded_path = uploads_dir / uploaded_name
    if source.is_symlink() or not source.is_file():
        raise RuntimeError('Drive file source должен быть regular file.')
    shutil.copy2(source, uploaded_path)
else:
    from google.colab import drive
    drive.mount('/content/drive')
    source = drive_path(PROJECT_SOURCE_PATH)
    if source.is_symlink() or not source.is_dir():
        raise RuntimeError('Drive folder source должен быть каталогом.')
    copy_drive_folder(source, project_dir)
    uploaded_path = None

if uploaded_path is not None:
    if not uploaded_name or uploaded_path.suffix.lower() not in {'.blend', '.zip'}:
        raise RuntimeError('Поддерживается только один файл .blend или .zip.')
    if uploaded_path.stat().st_size > MAX_PROJECT_INPUT_BYTES:
        raise RuntimeError('Project input превышает безопасный лимит размера.')
    if uploaded_path.suffix.lower() == '.zip':
        safe_extract_zip(uploaded_path, project_dir)
    else:
        project_dir.mkdir()
        shutil.copy2(uploaded_path, project_dir / uploaded_name)

BLEND_FILE = select_blend(project_dir)
validate_blend(BLEND_FILE)
print(f'Будет отрендерен: {BLEND_FILE}')

## Preflight

Сначала прочитайте JSON-отчёт. Проверка открывает проект отдельным read-only Blender process и сравнивает SHA-256 `.blend` до и после. Не запускайте full render при ошибках assets, camera или engine.

In [ ]:
#@title Read-only preflight проекта (JSON)
# Проверка запускает stock Blender с --factory-startup и --disable-autoexec, не вызывает bpy.ops и сверяет SHA-256 исходного .blend.
import hashlib
import json
import time

PREFLIGHT_SCRIPT = r'''import json
import os
from pathlib import Path
import shutil
import subprocess
import sys
import bpy

def issue(report, severity, code, message, **details):
    entry = {'code': code, 'message': message}
    if details:
        entry['details'] = details
    report['issues'][severity].append(entry)

def abspath(filepath, library=None):
    if not filepath:
        return ''
    try:
        return bpy.path.abspath(filepath, library=library)
    except TypeError:
        return bpy.path.abspath(filepath)

def add_asset(report, kind, block, filepath=None, packed=False):
    name = block.name
    if packed:
        report['assets'].append({'kind': kind, 'name': name, 'status': 'packed_in_blend', 'path': None})
        return
    path = abspath(filepath if filepath is not None else getattr(block, 'filepath', ''), getattr(block, 'library', None))
    if not path:
        report['assets'].append({'kind': kind, 'name': name, 'status': 'not_applicable', 'path': None})
        return
    exists = os.path.exists(path)
    report['assets'].append({'kind': kind, 'name': name, 'status': 'available' if exists else 'missing', 'path': path})
    if not exists:
        issue(report, 'errors', 'missing_asset', f'Missing {kind}: {name}', path=path)

def gpu_info():
    command = ['nvidia-smi', '--query-gpu=name,memory.total,memory.free', '--format=csv,noheader,nounits']
    try:
        result = subprocess.run(command, check=False, capture_output=True, text=True, timeout=10)
    except (OSError, subprocess.SubprocessError) as error:
        return {'available': False, 'devices': [], 'error': str(error)}
    if result.returncode:
        return {'available': False, 'devices': [], 'error': result.stderr.strip()}
    devices = []
    for line in result.stdout.splitlines():
        fields = [field.strip() for field in line.split(',')]
        if len(fields) == 3:
            devices.append({'name': fields[0], 'vram_total_mib': fields[1], 'vram_free_mib': fields[2]})
    return {'available': bool(devices), 'devices': devices}

def cycles_devices():
    addon = bpy.context.preferences.addons.get('cycles')
    if addon is None:
        return {'available': False, 'devices': []}
    try:
        preferences = addon.preferences
        refresh = getattr(preferences, 'refresh_devices', None)
        refresh() if refresh else preferences.get_devices()
        return {'available': True, 'devices': [{'name': device.name, 'type': str(device.type), 'enabled': bool(device.use)} for device in preferences.devices]}
    except Exception as error:
        return {'available': True, 'devices': [], 'error': str(error)}

def build_report(report_path):
    scenes = tuple(bpy.data.scenes)
    context_scene_name = getattr(getattr(bpy.context, 'scene', None), 'name', '')
    active_scene_name = context_scene_name if context_scene_name in {scene.name for scene in scenes} else scenes[0].name
    report = {'report_version': 1, 'source_file': bpy.data.filepath, 'blender': {'version': bpy.app.version_string, 'version_file': list(bpy.data.version)}, 'active_scene': active_scene_name, 'scenes': [], 'assets': [], 'issues': {'errors': [], 'warnings': []}, 'environment': {}}
    for scene in bpy.data.scenes:
        render = scene.render
        record = {'name': scene.name, 'camera': scene.camera.name if scene.camera else None, 'render_engine': render.engine, 'frame_start': scene.frame_start, 'frame_end': scene.frame_end, 'frame_step': scene.frame_step, 'fps': render.fps / render.fps_base, 'resolution': {'x': render.resolution_x, 'y': render.resolution_y, 'percentage': render.resolution_percentage}, 'output': {'filepath': abspath(render.filepath), 'file_format': render.image_settings.file_format, 'color_mode': render.image_settings.color_mode}}
        report['scenes'].append(record)
        if not record['camera']:
            issue(report, 'errors', 'missing_camera', f'Scene {scene.name!r} has no active camera.')
        if record['render_engine'] not in {'CYCLES', 'BLENDER_EEVEE', 'BLENDER_EEVEE_NEXT', 'BLENDER_WORKBENCH'}:
            issue(report, 'errors', 'unsupported_render_engine', f'Scene {scene.name!r} uses {record["render_engine"]!r}.')
        if record['frame_end'] < record['frame_start'] or record['frame_step'] < 1:
            issue(report, 'errors', 'invalid_frame_range', f'Scene {scene.name!r} has an invalid frame range.')
    for image in bpy.data.images:
        if image.source not in {'GENERATED', 'VIEWER'}:
            add_asset(report, 'image', image, packed=bool(getattr(image, 'packed_file', None)) or bool(getattr(image, 'packed_files', ())))
    for collection, kind in ((bpy.data.libraries, 'linked_library'), (bpy.data.fonts, 'font'), (bpy.data.volumes, 'vdb'), (bpy.data.cache_files, 'cache'), (bpy.data.movieclips, 'movie_clip'), (bpy.data.sounds, 'sound')):
        for block in collection:
            add_asset(report, kind, block)
    for obj in bpy.data.objects:
        for modifier in obj.modifiers:
            domain = getattr(modifier, 'domain_settings', None)
            cache_directory = getattr(domain, 'cache_directory', '')
            if cache_directory:
                add_asset(report, 'fluid_cache', obj, cache_directory)
    try:
        ram_total_bytes = os.sysconf('SC_PAGE_SIZE') * os.sysconf('SC_PHYS_PAGES')
    except (AttributeError, ValueError, OSError):
        ram_total_bytes = None
    disk = shutil.disk_usage(report_path.parent)
    system_root = Path(bpy.app.binary_path).resolve().parent
    external_addons = []
    for addon in bpy.context.preferences.addons:
        if addon.module == 'cycles':
            continue
        module_path = getattr(sys.modules.get(addon.module), '__file__', None)
        if not module_path:
            continue
        try:
            Path(module_path).resolve().relative_to(system_root)
            continue
        except ValueError:
            pass
        external_addons.append(addon.module)
    external_addons.sort()
    report['environment'] = {'ram_total_bytes': ram_total_bytes, 'disk_total_bytes': disk.total, 'disk_free_bytes': disk.free, 'gpu': gpu_info(), 'cycles_device_discovery': cycles_devices(), 'enabled_external_addons': external_addons}
    if external_addons:
        issue(report, 'warnings', 'external_addons_enabled', 'Enabled add-ons may be unavailable in stock Colab Blender.', modules=external_addons)
    return report

separator = sys.argv.index('--')
report_path = Path(sys.argv[separator + 1])
try:
    report = build_report(report_path)
except Exception as error:
    report = {'report_version': 1, 'source_file': bpy.data.filepath, 'blender': {'version': bpy.app.version_string}, 'active_scene': None, 'scenes': [], 'assets': [], 'issues': {'errors': [{'code': 'probe_failure', 'message': str(error)}], 'warnings': []}, 'environment': {}}
report_path.parent.mkdir(parents=True, exist_ok=True)
temporary_path = report_path.with_suffix(report_path.suffix + '.tmp')
temporary_path.write_text(json.dumps(report, ensure_ascii=False, indent=2, sort_keys=True), encoding='utf-8')
temporary_path.replace(report_path)
print(f'BLENDER_PREFLIGHT_REPORT={report_path}')
'''

def sha256_file(path):
    digest = hashlib.sha256()
    with path.open('rb') as source:
        for chunk in iter(lambda: source.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

preflight_dir = Path('/content/blender_preflight') / BLEND_FILE.stem
preflight_dir.mkdir(parents=True, exist_ok=True)
probe_script = preflight_dir / 'blender_preflight.py'
preflight_report_path = preflight_dir / 'preflight.json'
probe_script.write_text(PREFLIGHT_SCRIPT, encoding='utf-8')
source_hash_before_probe = sha256_file(BLEND_FILE)
probe_command = [str(BLENDER), '--background', '--factory-startup', '--disable-autoexec', str(BLEND_FILE), '--python', str(probe_script), '--', str(preflight_report_path)]
probe_result = subprocess.run(probe_command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
source_hash_after_probe = sha256_file(BLEND_FILE)
if source_hash_after_probe != source_hash_before_probe:
    raise RuntimeError('Read-only preflight изменил исходный .blend; дальнейший рендер заблокирован.')
if probe_result.returncode != 0 or not preflight_report_path.exists():
    raise RuntimeError(f'Blender preflight не завершился: {probe_result.stdout}')
PREFLIGHT_REPORT = json.loads(preflight_report_path.read_text(encoding='utf-8'))
required_keys = {'report_version', 'source_file', 'blender', 'scenes', 'assets', 'issues', 'environment'}
if PREFLIGHT_REPORT.get('report_version') != 1 or required_keys.difference(PREFLIGHT_REPORT):
    raise RuntimeError('Blender preflight вернул несовместимый JSON report.')
errors = PREFLIGHT_REPORT['issues']['errors']
warnings = PREFLIGHT_REPORT['issues']['warnings']
print(json.dumps(PREFLIGHT_REPORT, ensure_ascii=False, indent=2))
print(f'Preflight: {len(errors)} error(s), {len(warnings)} warning(s); SHA-256 source сохранён.')

if CONFIG.run_preflight_test_frame:
    if errors:
        raise RuntimeError('Test-frame заблокирован ошибками preflight; исправьте их до расходования GPU-времени.')
    active_scene = next(scene for scene in PREFLIGHT_REPORT['scenes'] if scene['name'] == PREFLIGHT_REPORT['active_scene'])
    test_frame = active_scene['frame_start']
    test_dir = preflight_dir / f'test_frame_{int(time.time())}'
    test_dir.mkdir()
    test_prefix = test_dir / 'frame_'
    print(f'Runtime-only test override: frame={test_frame}; output={test_prefix}; source .blend не сохраняется.')
    test_command = [str(BLENDER), '--background', '--factory-startup', '--disable-autoexec', str(BLEND_FILE)]
    if CONFIG.enable_cycles_gpu:
        test_gpu_script = test_dir / 'configure_cycles_gpu.py'
        test_gpu_script.write_text(f'''import bpy
ALLOW_CPU_FALLBACK = {CONFIG.allow_cpu_fallback!r}
INCLUDE_CPU_WITH_GPU = {CONFIG.include_cpu_with_gpu!r}
addon = bpy.context.preferences.addons.get('cycles')
cycles_scenes = [scene for scene in bpy.data.scenes if scene.render.engine == 'CYCLES']
if cycles_scenes:
    if addon is None:
        raise RuntimeError('Cycles preferences are unavailable.')
    preferences = addon.preferences
    failures = []
    for backend in ('OPTIX', 'CUDA'):
        try:
            preferences.compute_device_type = backend
            refresh = getattr(preferences, 'refresh_devices', None)
            refresh() if refresh else preferences.get_devices()
            devices = tuple(preferences.devices)
            selected = [device for device in devices if str(device.type).upper() == backend]
            if not selected:
                failures.append(f'{{backend}}: no matching Cycles devices')
                continue
            for device in devices:
                device.use = str(device.type).upper() == backend or (INCLUDE_CPU_WITH_GPU and str(device.type).upper() == 'CPU')
            for scene in cycles_scenes:
                scene.cycles.device = 'GPU'
            print(f'CYCLES_DEVICE: backend={{backend}}')
            break
        except Exception as error:
            failures.append(f'{{backend}}: {{error}}')
    else:
        if not ALLOW_CPU_FALLBACK:
            raise RuntimeError('Cycles GPU was not configured. CPU fallback is disabled. ' + '; '.join(failures))
        for scene in cycles_scenes:
            scene.cycles.device = 'CPU'
''', encoding='utf-8')
        test_command += ['--python', str(test_gpu_script)]
    test_command += ['-o', str(test_prefix), '-f', str(test_frame)]
    source_hash_before_test = sha256_file(BLEND_FILE)
    started_at = time.monotonic()
    try:
        subprocess.run(test_command, check=True)
    finally:
        if sha256_file(BLEND_FILE) != source_hash_before_test:
            raise RuntimeError('Test-frame изменил исходный .blend; дальнейший рендер заблокирован.')
    elapsed_seconds = time.monotonic() - started_at
    test_output_bytes = sum(path.stat().st_size for path in test_dir.rglob('*') if path.is_file())
    frame_count = ((active_scene['frame_end'] - active_scene['frame_start']) // active_scene['frame_step']) + 1
    estimate = {'frame_count': frame_count, 'test_frame_seconds': elapsed_seconds, 'test_frame_output_bytes': test_output_bytes, 'estimated_render_seconds': elapsed_seconds * frame_count, 'estimated_output_bytes': test_output_bytes * frame_count, 'note': 'Линейная оценка по одному кадру; сложность кадров и startup Blender могут отличаться.'}
    (test_dir / 'estimate.json').write_text(json.dumps(estimate, ensure_ascii=False, indent=2), encoding='utf-8')
    print(json.dumps(estimate, ensure_ascii=False, indent=2))
else:
    print('Test-frame не запускался. Установите RUN_PREFLIGHT_TEST_FRAME=True только после просмотра JSON report.')

## Рендер и resume

Подключите Drive для durable job results. Оставьте `JOB_ID` пустым для нового задания. После interruption повторите upload и preflight, вставьте прежний UUID и notebook отрендерит только checksum-подтверждённые отсутствующие кадры.

<details><summary>Runtime-only override</summary>

Для resumable sequence временно меняется только output path на локальный `/content` staging. Камеры, движок, качество и другие сохранённые scene settings не меняются и исходный `.blend` не сохраняется.

</details>

In [ ]:
#@title Результаты: Drive или локальное скачивание
RESULT_DESTINATION = 'drive' #@param ['drive', 'local_download']
DRIVE_RESULT_PATH = 'Blender Renders' #@param {type: 'string'}
from datetime import datetime

if RESULT_DESTINATION == 'drive':
    from google.colab import drive
    drive.mount('/content/drive')
    if not isinstance(DRIVE_RESULT_PATH, str) or not DRIVE_RESULT_PATH.strip():
        raise RuntimeError('DRIVE_RESULT_PATH должен быть непустым путём внутри MyDrive.')
    drive_root = Path('/content/drive/MyDrive').resolve()
    DRIVE_OUTPUT_DIR = (drive_root / DRIVE_RESULT_PATH.strip()).resolve()
    try:
        DRIVE_OUTPUT_DIR.relative_to(drive_root)
    except ValueError as error:
        raise RuntimeError('DRIVE_RESULT_PATH должен оставаться внутри MyDrive.') from error
elif RESULT_DESTINATION == 'local_download':
    DRIVE_OUTPUT_DIR = Path('/content/blender_local_results')
else:
    raise RuntimeError('RESULT_DESTINATION должен быть drive или local_download.')
DRIVE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f'Результаты будут сохранены в: {DRIVE_OUTPUT_DIR}')

In [ ]:
#@title Выбор устройства Cycles и запуск рендера
# Скрипт применяется только к процессу Blender; исходный .blend не сохраняется.
gpu_script = Path('/content/configure_cycles_gpu.py')
gpu_script.write_text(f'''import bpy

ALLOW_CPU_FALLBACK = {CONFIG.allow_cpu_fallback!r}
INCLUDE_CPU_WITH_GPU = {CONFIG.include_cpu_with_gpu!r}

def refresh_devices(preferences):
    refresh = getattr(preferences, 'refresh_devices', None)
    if refresh is not None:
        refresh()
        return
    legacy_get_devices = getattr(preferences, 'get_devices', None)
    if legacy_get_devices is None:
        raise RuntimeError('Cycles preferences do not expose device discovery.')
    legacy_get_devices()

def configure_cycles():
    cycles_scenes = [scene for scene in bpy.data.scenes if scene.render.engine == 'CYCLES']
    if not cycles_scenes:
        print('CYCLES_DEVICE: no Cycles scenes; no device selection required.')
        return
    addon = bpy.context.preferences.addons.get('cycles')
    if addon is None:
        raise RuntimeError('Cycles add-on/preferences are unavailable.')
    preferences = addon.preferences
    failures = []
    for backend in ('OPTIX', 'CUDA'):
        try:
            preferences.compute_device_type = backend
            refresh_devices(preferences)
            devices = tuple(preferences.devices)
            selected_gpus = [device for device in devices if str(device.type).upper() == backend]
            if not selected_gpus:
                failures.append(f'{{backend}}: no matching Cycles devices')
                continue
            for device in devices:
                device_type = str(device.type).upper()
                device.use = device_type == backend or (INCLUDE_CPU_WITH_GPU and device_type == 'CPU')
            for scene in cycles_scenes:
                scene.cycles.device = 'GPU'
            active = [f'{{device.name}} ({{device.type}})' for device in devices if device.use]
            print(f'CYCLES_DEVICE: backend={{backend}}; active={{active}}')
            return
        except Exception as error:
            failures.append(f'{{backend}}: {{error}}')
    if ALLOW_CPU_FALLBACK:
        for scene in cycles_scenes:
            scene.cycles.device = 'CPU'
        print('CYCLES_DEVICE: CPU fallback explicitly allowed; ' + '; '.join(failures))
        return
    raise RuntimeError('Cycles GPU was not configured. CPU fallback is disabled. ' + '; '.join(failures))

configure_cycles()
''', encoding='utf-8')

# Protocol v1: оставьте JOB_ID пустым для нового UUID; при resume вставьте
# напечатанный ранее id. Подтверждённые frames не перерендерятся.
JOB_ID = '' #@param {type: 'string'}
FRAME_CHUNK_SIZE = 10 #@param {type: 'integer'}

import hashlib
import os
import uuid
from pathlib import PurePosixPath

PROTOCOL_VERSION = '1.0'
WORKER_VERSION = 'notebook-0.1.0'
if not isinstance(JOB_ID, str):
    raise ValueError('JOB_ID должен быть строкой UUID или пустой строкой.')
if not isinstance(FRAME_CHUNK_SIZE, int) or isinstance(FRAME_CHUNK_SIZE, bool) or FRAME_CHUNK_SIZE < 1:
    raise ValueError('FRAME_CHUNK_SIZE должен быть положительным целым числом.')
job_id = JOB_ID.strip() or str(uuid.uuid4())
try:
    parsed_job_id = uuid.UUID(job_id)
except ValueError as error:
    raise ValueError('JOB_ID должен быть UUID v4.') from error
if parsed_job_id.version != 4:
    raise ValueError('JOB_ID должен быть UUID v4.')

def protocol_now():
    return datetime.now().astimezone().isoformat(timespec='seconds')

def atomic_json(path, value):
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + '.tmp')
    temporary.write_text(json.dumps(value, ensure_ascii=False, indent=2, sort_keys=True) + '\n', encoding='utf-8')
    os.replace(temporary, path)

def file_sha256(path):
    digest = hashlib.sha256()
    with path.open('rb') as source:
        for data in iter(lambda: source.read(1024 * 1024), b''):
            digest.update(data)
    return digest.hexdigest()

active_scene = next(scene for scene in PREFLIGHT_REPORT['scenes'] if scene['name'] == PREFLIGHT_REPORT['active_scene'])
if CONFIG.render_mode is RenderMode.STILL:
    requested_frames = (CONFIG.still_frame,)
    render_mode = 'single_frame'
else:
    if active_scene['output']['file_format'] == 'FFMPEG':
        raise RuntimeError('Resumable animation требует image sequence; FFMPEG/video output не имеет проверяемых кадров.')
    requested_frames = tuple(range(active_scene['frame_start'], active_scene['frame_end'] + 1, active_scene['frame_step']))
    render_mode = 'saved_range'

job_root = DRIVE_OUTPUT_DIR / 'jobs' / job_id
request_dir, worker_dir, result_dir = job_root / 'request', job_root / 'worker', job_root / 'result'
job_path, status_path = request_dir / 'job.json', worker_dir / 'status.json'
manifest_path, log_path, summary_path = result_dir / 'result_manifest.json', worker_dir / 'render.log', worker_dir / 'summary.json'
source_sha256 = file_sha256(BLEND_FILE)
if job_path.exists():
    job = json.loads(job_path.read_text(encoding='utf-8'))
    if job.get('protocol_version') != PROTOCOL_VERSION or job.get('job_id') != job_id:
        raise RuntimeError('Существующий job.json несовместим с protocol v1 или JOB_ID.')
    if job.get('project', {}).get('sha256') != source_sha256:
        raise RuntimeError('Загруженный .blend не совпадает с immutable job.json; выберите новый JOB_ID.')
    expected_render = {'mode': render_mode, 'start': requested_frames[0], 'end': requested_frames[-1], 'step': active_scene['frame_step'] if render_mode == 'saved_range' else 1}
    if any(job.get('render', {}).get(key) != value for key, value in expected_render.items()):
        raise RuntimeError('Текущие render settings не совпадают с immutable job.json; выберите новый JOB_ID.')
else:
    job = {'protocol_version': PROTOCOL_VERSION, 'job_id': job_id, 'created_at': protocol_now(), 'addon_version': WORKER_VERSION, 'project': {'display_name': BLEND_FILE.stem, 'archive': BLEND_FILE.name, 'sha256': source_sha256, 'size_bytes': BLEND_FILE.stat().st_size}, 'blender': {'source_version': PREFLIGHT_REPORT['blender']['version'], 'requested_version': BLENDER_VERSION}, 'render': {'mode': render_mode, 'start': requested_frames[0], 'end': requested_frames[-1], 'step': active_scene['frame_step'] if render_mode == 'saved_range' else 1, 'overwrite': False, 'use_saved_settings': True, 'overrides': {'output': {'mode': 'runtime_staging', 'reason': 'resumable checksum-verified frame delivery'}}}, 'destination': {'mode': 'local'}}
    atomic_json(job_path, job)

def append_log(message):
    worker_dir.mkdir(parents=True, exist_ok=True)
    with log_path.open('a', encoding='utf-8') as log:
        log.write(f'{protocol_now()} {message}\n')
        log.flush()

previous_status = json.loads(status_path.read_text(encoding='utf-8')) if status_path.exists() else None
def publish_status(state, completed, message, current_frame=None, error=None):
    global previous_status
    allowed = {'PREFLIGHT': {'RENDERING', 'FAILED'}, 'RENDERING': {'RENDERING', 'PARTIAL', 'COMPLETE', 'FAILED'}, 'PARTIAL': {'PREFLIGHT', 'RENDERING', 'FAILED'}, 'FAILED': {'PREFLIGHT'}, 'COMPLETE': {'PREFLIGHT'}}
    if previous_status is not None and state not in allowed[previous_status['state']]:
        raise RuntimeError(f'Недопустимый переход status: {previous_status["state"]} -> {state}')
    status = {'protocol_version': PROTOCOL_VERSION, 'job_id': job_id, 'revision': 1 if previous_status is None else previous_status['revision'] + 1, 'state': state, 'updated_at': protocol_now(), 'worker_version': WORKER_VERSION, 'progress': {'completed_frames': completed, 'total_frames': len(requested_frames), 'current_frame': current_frame}, 'device': {}, 'message': message, 'error': error}
    atomic_json(status_path, status)
    previous_status = status

def publish_manifest(records, complete):
    manifest = {'protocol_version': PROTOCOL_VERSION, 'job_id': job_id, 'published_at': protocol_now(), 'state': 'COMPLETE' if complete else 'PARTIAL', 'artifacts': [records[frame] for frame in sorted(records)]}
    atomic_json(manifest_path, manifest)
    return manifest

def safe_result_path(value):
    if not isinstance(value, str):
        return None
    candidate = PurePosixPath(value)
    if candidate.is_absolute() or '..' in candidate.parts:
        return None
    return result_dir.joinpath(*candidate.parts)

records = {}
if manifest_path.exists():
    existing = json.loads(manifest_path.read_text(encoding='utf-8'))
    if existing.get('protocol_version') != PROTOCOL_VERSION or existing.get('job_id') != job_id:
        raise RuntimeError('Существующий result_manifest.json несовместим с job.')
    for record in existing.get('artifacts', []):
        if not isinstance(record, dict):
            continue
        target = safe_result_path(record.get('path', ''))
        if record.get('frame') in requested_frames and target is not None and target.is_file() and target.stat().st_size == record.get('size_bytes') and file_sha256(target) == record.get('sha256'):
            records[record['frame']] = record
missing_frames = tuple(frame for frame in requested_frames if frame not in records)
chunks, current = [], []
step = job['render']['step']
for frame in missing_frames:
    if current and (frame != current[-1] + step or len(current) == FRAME_CHUNK_SIZE):
        chunks.append(tuple(current)); current = []
    current.append(frame)
if current:
    chunks.append(tuple(current))

def publish_summary(state, error=None):
    atomic_json(summary_path, {'protocol_version': PROTOCOL_VERSION, 'job_id': job_id, 'published_at': protocol_now(), 'state': state, 'verified_frames': len(records), 'total_frames': len(requested_frames), 'result_manifest': '../result/result_manifest.json', 'error': error})

append_log(f'JOB_START missing_frames={len(missing_frames)}')
print('Runtime-only output override: локальный staging -> checksum-verified result/frames; source .blend settings не сохраняются.')
try:
    reuse_complete = not chunks and previous_status is not None and previous_status['state'] == 'COMPLETE'
    if reuse_complete:
        append_log('JOB_COMPLETE reused_verified_frames=true')
    else:
        publish_status('PREFLIGHT', len(records), 'Планирование missing frames.')
        if not chunks:
            publish_status('RENDERING', len(records), 'Подтверждённые кадры готовы к completion без нового Blender process.')
    for chunk in chunks:
        publish_status('RENDERING', len(records), f'Рендер frames {chunk[0]}-{chunk[-1]}.', chunk[0])
        chunk_dir = Path('/content/blender_render_chunks') / job_id / f'{chunk[0]}-{chunk[-1]}'
        chunk_dir.mkdir(parents=True, exist_ok=True)
        command = [str(BLENDER), '-b', str(BLEND_FILE)]
        if CONFIG.enable_cycles_gpu: command += ['-P', str(gpu_script)]
        command += ['-o', str(chunk_dir / 'frame_'), '-s', str(chunk[0]), '-e', str(chunk[-1]), '-j', str(step), '-a']
        append_log(f'CHUNK_START frames={chunk}')
        source_before_render = file_sha256(BLEND_FILE)
        process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
        for line in process.stdout:
            append_log('BLENDER ' + line.rstrip())
        if process.wait() != 0:
            raise RuntimeError(f'Blender завершился с exit code {process.returncode}; смотрите {log_path}.')
        if file_sha256(BLEND_FILE) != source_before_render:
            raise RuntimeError('Рендер изменил исходный .blend; дальнейшая публикация заблокирована.')
        for frame in chunk:
            matches = sorted(chunk_dir.glob(f'frame_{frame:04d}.*'))
            if len(matches) != 1:
                raise RuntimeError(f'Не найден ровно один output для frame {frame}: {matches}')
            target = result_dir / 'frames' / matches[0].name
            target.parent.mkdir(parents=True, exist_ok=True)
            temporary = target.with_suffix(target.suffix + '.tmp')
            shutil.copyfile(matches[0], temporary)
            digest, size = file_sha256(temporary), temporary.stat().st_size
            os.replace(temporary, target)
            records[frame] = {'frame': frame, 'path': f'frames/{target.name}', 'sha256': digest, 'size_bytes': size}
            publish_manifest(records, complete=False)
        append_log(f'CHUNK_COMPLETE frames={chunk}')
    manifest = publish_manifest(records, complete=True)
    if not reuse_complete:
        publish_status('COMPLETE', len(records), 'Все кадры checksum-verified.')
    publish_summary('COMPLETE')
    output_dir = result_dir / 'frames'
    print(f'JOB_COMPLETE job_id={job_id}; frames={len(records)}/{len(requested_frames)}; manifest={manifest_path}')
except KeyboardInterrupt:
    publish_manifest(records, complete=False); publish_status('PARTIAL', len(records), 'Runtime прерван; verified frames готовы к resume.'); publish_summary('PARTIAL', {'type': 'KeyboardInterrupt'})
    append_log('JOB_PARTIAL reason=KeyboardInterrupt')
    raise
except Exception as error:
    publish_manifest(records, complete=False); publish_status('FAILED', len(records), 'Render failed; verified frames сохранены.', error={'type': type(error).__name__, 'message': str(error)}); publish_summary('FAILED', {'type': type(error).__name__, 'message': str(error)})
    append_log(f'JOB_FAILED type={type(error).__name__} message={error}')
    raise

## Результаты

Файлы уже checksum-verified в Drive job folder. Следующая ячейка показывает их и, только при включённом Advanced-параметре `DOWNLOAD_RESULT`, скачивает один файл либо ZIP.

In [ ]:
#@title Результаты: ZIP, preview и опциональный MP4
CREATE_RESULT_ZIP = True #@param {type: 'boolean'}
BUILD_RESULT_MP4 = False #@param {type: 'boolean'}
MP4_FPS = 24 #@param {type: 'integer'}
from google.colab import files
from IPython.display import Image, display
import subprocess

if not isinstance(CREATE_RESULT_ZIP, bool) or not isinstance(BUILD_RESULT_MP4, bool) or isinstance(MP4_FPS, bool) or not isinstance(MP4_FPS, int) or MP4_FPS < 1:
    raise RuntimeError('Параметры результатов имеют недопустимые значения.')
rendered_files = sorted(path for path in output_dir.rglob('*') if path.is_file())
if not rendered_files:
    raise RuntimeError('Рендер завершился, но файлов в папке результата нет.')

print(f'Готово: {len(rendered_files)} файл(ов) в {output_dir}')
preview_candidates = [path for path in rendered_files if path.suffix.lower() in {'.png', '.jpg', '.jpeg', '.webp'}]
if preview_candidates:
    preview = preview_candidates[-1]
    print(f'Preview: {preview.name}')
    display(Image(filename=str(preview)))

result_zip = None
if CREATE_RESULT_ZIP or RESULT_DESTINATION == 'local_download':
    result_base = output_dir.parent / output_dir.name
    result_zip = Path(shutil.make_archive(str(result_base), 'zip', output_dir))
    print(f'ZIP: {result_zip.name} ({result_zip.stat().st_size / 1024 / 1024:.1f} MB)')

mp4_path = None
if BUILD_RESULT_MP4:
    frame_dir = output_dir
    png_frames = sorted(frame_dir.glob('*.png'))
    if not png_frames:
        raise RuntimeError('MP4 требует PNG image sequence из текущего результата.')
    mp4_path = output_dir.parent / f'{output_dir.name}.mp4'
    subprocess.run(['ffmpeg', '-y', '-framerate', str(MP4_FPS), '-pattern_type', 'glob', '-i', str(frame_dir / '*.png'), '-c:v', 'libx264', '-pix_fmt', 'yuv420p', str(mp4_path)], check=True)
    print(f'MP4: {mp4_path.name} ({mp4_path.stat().st_size / 1024 / 1024:.1f} MB)')

if CONFIG.download_result or RESULT_DESTINATION == 'local_download':
    result = mp4_path or result_zip or rendered_files[0]
    print(f'Скачивается: {result.name} ({result.stat().st_size / 1024 / 1024:.1f} MB)')
    files.download(str(result))